## Preliminares

In [1]:
# Importar modulos y funciones necesarias
import pandas as pd
import numpy as np
from datetime import datetime
from statsmodels.formula.api import ols
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from funciones import *

In [2]:
# Abrir archivo clean_data
data_folder = "../data"
df = pd.read_parquet(f"{data_folder}/clean_data.parquet")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   YearsSinceAdded                 503 non-null    float64 
 1   Beta                            503 non-null    float64 
 2   DividendYield                   503 non-null    float64 
 3   ReturnOnAssets                  503 non-null    float64 
 4   profitMargins                   503 non-null    float64 
 5   operatingMargins                503 non-null    float64 
 6   currentRatio                    503 non-null    float64 
 7   revenueGrowth                   503 non-null    float64 
 8   shortPercentOfFloat             503 non-null    float64 
 9   PriceToBook_Transformed         503 non-null    float64 
 10  returnOnEquity_Transformed      503 non-null    float64 
 11  ForwardPE_Transformed           503 non-null    float64 
 12  MarketCap_log         

# Modelo de ensamblado de árboles RandomForest

In [ ]:
# Ratios de valuación
ratios = ['PriceToBook_Transformed', 'ForwardPE_Transformed', 'trailingPegRatio_log', 'EnterpriseToEbitda_Transformed']

# Se define la variable objetivo y las variables predictoras
label = 'EnterpriseToEbitda_Transformed'
X = df.drop(columns=ratios + ['Ticker']) # Se quitan los ratios de valuación y Ticker de las variables predictoras
y = df[label]

# identificar columnas numéricas y categóricas
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()

# preprocesador: escala numéricas y codifica categóricas
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

pipe = Pipeline([
    ('pre', preprocessor),
    ('model', RandomForestRegressor(
        random_state=42,
        n_estimators=300,
        max_depth=25,
        min_samples_leaf=1,
        max_features='sqrt',
        max_samples= None        
        ))
])

pipe.fit(X_train, y_train)

# comparar predicciones con el conjunto test

y_pred = pipe.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MSE: 1.0277986386988485
R2: 0.18830252730677344


In [4]:
# Reajustar modelo con todos los datos
print("Reajustando modelo con el dataset completo...")

X_completo = df.drop([label,'Ticker'], axis=1)
y_completo = df[label]

pipe_final = Pipeline([
    ('pre', preprocessor),
    ('model', RandomForestRegressor(
        random_state=42,
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=3,
        min_samples_split=5,
        max_features=0.5,
        max_samples= 0.7        
        ))
])

pipe_final.fit(X_completo, y_completo)
r2_final = pipe_final.score(X_completo, y_completo)
print(f"R² sobre dataset completo: {r2_final:.4f}")

Reajustando modelo con el dataset completo...
R² sobre dataset completo: 0.7398


In [5]:
# Test de validación cruzada
cross_val_scores = cross_val_score(pipe_final, X_completo, y_completo, cv=5, scoring='r2')
print(f"R² promedio CV: {cross_val_scores.mean():.4f} ± {cross_val_scores.std():.4f}")

R² promedio CV: 0.4313 ± 0.1099


* Se detecta un gran sobre-ajuste del modelo, con una disminución del ajuste del modelo en validación cruzada.
* Al ser una muestra pequena con solo 500 observaciones, se optimizan los hiperparametros reduciendo la maxima profundidad de los arboles, y aumentar el minimo de ejemplos por rama para evitar que el modelo *memorice* a cada accion del dataset.

In [6]:
# Importancia de factores en el modelo
rf_model = pipe_final.named_steps['model']
importances = rf_model.feature_importances_

# Obtener los nombres de las características después del preprocesamiento
preprocessor = pipe_final.named_steps['pre']
feature_names = preprocessor.get_feature_names_out()

feature_importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)
feature_importance_df.head(20)

,feature,importance
7,num__revenueGrowth,0.215966
1,num__Beta,0.166195
10,num__MarketCap_log,0.158145
2,num__DividendYield,0.141012
8,num__shortPercentOfFloat,0.051762
3,num__ReturnOnAssets,0.048149
11,num__debtToEquity_log,0.047363
9,num__returnOnEquity_Transformed,0.036993
5,num__operatingMargins,0.035310
0,num__YearsSinceAdded,0.034768


## Aplicacion del modelo

Se generan 2 clusters segun las predicciones:
* Overprediction: si los residuos son mayores o iguales a cero.
*  Underprediction: si los residuos son menores a cero.

In [7]:
# Cluster: sobre/sub prediccion respecto a la identidad
residuos = y_pred - y_test
cluster = pd.Series(['Overprediction' if r >= 0 else 'Underprediction' for r in residuos], 
                     index=y_test.index)

# Visualizar con los 2 clusters
fig = px.scatter(x=y_test, y=y_pred, color=cluster,
                 labels={'x':'Real','y':'Predicho','color':'Cluster'},
                 title='Predicciones vs Reales (Clusters)')
fig.add_shape(type='line', x0=y_test.min(), y0=y_test.min(),
             x1=y_test.max(), y1=y_test.max(),
             line=dict(color='black', dash='dash', width=2))
fig.show()

# Estadísticas por cluster
print("\nEstadísticas por cluster:")
print(f"Overprediction: {(cluster == 'Overprediction').sum()} casos, residuo medio: {residuos[cluster == 'Overprediction'].mean():.4f}")
print(f"Underprediction: {(cluster == 'Underprediction').sum()} casos, residuo medio: {residuos[cluster == 'Underprediction'].mean():.4f}")


Estadísticas por cluster:
Overprediction: 66 casos, residuo medio: 0.5783
Underprediction: 60 casos, residuo medio: -0.5227


In [8]:
# Recrear clusters con predicciones del modelo final
y_pred_completo = pipe_final.predict(X_completo)
residuos_completo = y_pred_completo - y_completo
cluster_completo = pd.Series(['Overprediction' if r > 0 else 'Underprediction' for r in residuos_completo], 
                              index=y_completo.index)

# Actualizar columna de cluster y calcular distancia a la identidad
df['cluster_prediction'] = cluster_completo.values
# Distancia perpendicular a la recta identidad (y = x)
df['distance_identity'] = residuos_completo / np.sqrt(2)

# Visualizar con plotly
fig = px.scatter(df, x=label, y=y_pred_completo, color='cluster_prediction',
                 hover_data=['Ticker','Sector'],
                 labels={'x':'Label (Real)','y':'Label (Predicho)','cluster_prediction':'Cluster'},
                 title='Clusters Overprediction/Underprediction (Datos Completos)')
fig.add_shape(type='line', x0=y_completo.min(), y0=y_completo.min(),
             x1=y_completo.max(), y1=y_completo.max(),
             line=dict(color='black', dash='dash', width=2))
fig.show()

# Estadísticas por cluster
print("\nEstadísticas de clusters (datos completos):")
print(f"Overprediction: {(df['cluster_prediction'] == 'Overprediction').sum()} casos, residuo medio: {residuos_completo[cluster_completo == 'Overprediction'].mean():.4f}")
print(f"Underprediction: {(df['cluster_prediction'] == 'Underprediction').sum()} casos, residuo medio: {residuos_completo[cluster_completo == 'Underprediction'].mean():.4f}")


Estadísticas de clusters (datos completos):
Overprediction: 279 casos, residuo medio: 0.2734
Underprediction: 224 casos, residuo medio: -0.3398


In [9]:
print("\nDataframe con cluster:")
print(df[['Ticker', label, 'distance_identity','cluster_prediction']].sort_values('distance_identity', ascending=False).head(10))


Dataframe con cluster:
    Ticker  EnterpriseToEbitda_Transformed  distance_identity  \
69      BA                       -6.330856           3.988242   
317   MRNA                       -2.108437           1.598748   
309   META                       -0.284415           0.990743   
61   BRK-B                       -1.712358           0.953344   
88    CVNA                        0.336532           0.728879   
145   DELL                        0.275038           0.706039   
124    CEG                       -0.272071           0.685475   
331    NEM                       -0.948603           0.681726   
5     ADBE                       -0.600672           0.676361   
198   FSLR                       -0.331271           0.649799   

    cluster_prediction  
69      Overprediction  
317     Overprediction  
309     Overprediction  
61      Overprediction  
88      Overprediction  
145     Overprediction  
124     Overprediction  
331     Overprediction  
5       Overprediction  
198     Ov

In [10]:
# Guardar dataframe con cluster con fecha en el nombre del archivo

dia = datetime.now().day
mes = datetime.now().month
year = datetime.now().year

df.to_csv(f"{data_folder}/cluster_data_{year}_{mes}_{dia}.csv", index=False)

## Anexo: optimizacion de hiperparametros

* La siguiente celda se utilizó para optimizar los hiperparámetros de los modelos de forma individual. La ejecución de la misma no es necesaria.
* En caso de utilizarla, reemplazar 'no' por 'si' en la primer línea. Puede demorar varios minutos.

In [11]:
ejecutar_celda = 'no' # Reemplazar para ejecutar
modelo_a_optimizar = 1  # 1 = Random Forest 

if ejecutar_celda == 'si':

    if modelo_a_optimizar == 1:
        nombre_modelo= 'Random Forest'
        print(f"Configurando GridSearchCV para {nombre_modelo}")

        # Pipeline usando el preprocesador específico para Random Forest
        modelo_base = Pipeline(steps=[
            ('preprocesador', preprocessor),
            ('rf', RandomForestRegressor(random_state=42))
        ])

        param_grid = {
            'rf__n_estimators': [300],
            'rf__max_depth': [10],
            'rf__min_samples_leaf': [3],
            'rf__min_samples_split': [5],
            'rf__max_features': ['sqrt', 'log2', 0.5],
            'rf__max_samples': [0.7]
        }

    elif modelo_a_optimizar == 2: # Por implementar
        nombre_modelo= 'LightGBM'
        print(f"Configurando GridSearchCV para {nombre_modelo}")

        # Pipeline usando el preprocesador específico para LightGBM
        modelo_base = Pipeline(steps=[
            ('preprocesador', preprocesado_rf),
            ('lgb', lgb.LGBMRegressor(random_state=42,
                                      subsample=0.8,
                                      colsample_bytree=0.8
))
        ])

        param_grid = {
            'lgb__n_estimators': [500, 550, 600],
            'lgb__max_depth': [5],
            'lgb__learning_rate': [0.05],
            'lgb__num_leaves': [20],
            'lgb__min_child_samples': [20]
        }

    else:
        raise ValueError("Opción no válida. Por favor, selecciona 1 o 2.")

    # Configurar el GridSearchCV
    grid_search = GridSearchCV(
        estimator=modelo_base,
        param_grid=param_grid,
        scoring='neg_mean_absolute_error',
        cv=3,
        n_jobs=-1,
        verbose=2
    )

    # Entrenar con X_train y y_train originales
    print(f"Iniciando búsqueda de hiperparámetros. Esto tomará unos minutos.")
    grid_search.fit(X_train, y_train)

    # Resultados
    print("\n--- Búsqueda Finalizada ---")
    print("Mejores hiperparámetros encontrados:")
    print(grid_search.best_params_)

    # Extraer el mejor modelo
    best_model = grid_search.best_estimator_

    # Predecir en el set de prueba usando X_test original
    y_pred = best_model.predict(X_test)

    print("\nEvaluación en el set de Prueba:\n", obtener_metricas(y_test, y_pred, nombre_modelo))